In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


In [ ]:
# --- Data Directory Setup (auto-generated) ---
from pathlib import Path
import os

# Resolve data directory relative to workspace root
def _find_data_dir():
    """Find the data directory by walking up from notebook location."""
    candidates = [
        Path.cwd().parent / "data" / "NLP Projecct 16.NLP for whatsapp chats",
        Path.cwd() / "data" / "NLP Projecct 16.NLP for whatsapp chats",
        Path(".").resolve().parent / "data" / "NLP Projecct 16.NLP for whatsapp chats",
    ]
    for c in candidates:
        if c.exists():
            return c
    # Fallback: current directory
    return Path(".")

DATA_DIR = _find_data_dir()
print(f"Data directory: {DATA_DIR}")

In [155]:
#Importing pandas and numpy library
import numpy as np 
import pandas as pd

In [156]:
#Loading the sad happy and angry datasets
sad = pd.read_csv(str(DATA_DIR / 'sad.csv'))
happy= pd.read_csv(str(DATA_DIR / 'happy.csv'))
angry=pd.read_csv(str(DATA_DIR / 'angry.csv'))

In [157]:
#Finding the shapes of all datas
print("Happy : ",happy.shape)
print("Angry : ",angry.shape)
print("Sad : ",sad.shape)

Happy :  (708, 2)
Angry :  (696, 2)
Sad :  (635, 2)


In [158]:
happy.head(2)

,content,sentiment
0,Wants to know how the hell I can remember word...,happy
1,Love is a long sweet dream & marriage is an al...,happy


In [159]:
angry.head(2)

,content,sentiment
0,"Sometimes I’m not angry, I’m hurt and there’s ...",angry
1,Not available for busy people☺,angry


In [160]:
sad.head(2)

,content,sentiment
0,"Never hurt people who love you a lot, because ...",sad
1,Don’t expect me to tell you what you did wrong...,sad


In [161]:
#Dropping the duplicates from all the datas
happy = happy.drop_duplicates(subset='content', keep="first")
angry = angry.drop_duplicates(subset='content', keep="first")
sad = sad.drop_duplicates(subset='content', keep="first")

In [162]:
#Concatenating the three datas in to df
frames = [sad, happy, angry]
df = pd.concat(frames)

In [163]:
#Displaying the concatenated dataframe
df.head(3)

,content,sentiment
0,"Never hurt people who love you a lot, because ...",sad
1,Don’t expect me to tell you what you did wrong...,sad
2,I preferred walking away than fighting for you...,sad


In [164]:
#Looking for the info of df
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1592 entries, 0 to 694
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   content    1592 non-null   object
 1   sentiment  1592 non-null   object
dtypes: object(2)
memory usage: 37.3+ KB


In [165]:
#Dropping the duplicates
df = df.drop_duplicates(subset='content', keep="first")

In [166]:
#Checking for count on each unique item
df['sentiment'].value_counts()

happy    702
angry    494
sad      390
Name: sentiment, dtype: int64

In [167]:
#replacing the categorical values
df['sentiment'].replace({'happy':1,'angry':0,'sad':2},inplace=True)

In [168]:
#Categorical converted to numerial
df['sentiment'].value_counts()

1    702
0    494
2    390
Name: sentiment, dtype: int64

In [169]:
import re
#Defining function to clean the html tag from text
def html(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '',text)
df['content']=df['content'].apply(html)

In [170]:
## Defining function to convert to lower case
def lower(text):
    return text.lower()
df['content']=df['content'].apply(lower)

In [171]:
#defining Function to remove special characters
def special(text):
        x=''
        for i in text:
            if i.isalnum():
                x=x+i
            else:
                x=x+' '
        return x
df['content']=df['content'].apply(special)

In [172]:
#Importing NLTK library
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import nltk
from nltk.stem.porter import PorterStemmer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [173]:
#defining function to remove stopwords using nltk library
def r_stopwords(text):
    x=[]
    for i in text.split():
        
        if i not in stopwords.words('english'):
            x.append(i)
    y=x[:]
    x.clear()
    return y
df['content']=df['content'].apply(r_stopwords)

In [174]:
#Defining function to join back the list of elements
def join_back(list_input):
    return " ".join(list_input)
df['content']=df['content'].apply(join_back)

In [175]:
#instantiation of portstemmer
ps= PorterStemmer()
y=[]
#Defining function to stemming the words
def stem_words(text):
    for i in text:
        y.append(ps.stem(i))
    z=y[:]
    y.clear()
    return z
df['content']=df['content'].apply(stem_words)

In [176]:
#Defining a joinback function to join the items
def joinback2(list_input):
    return "".join(list_input)
df['content']=df['content'].apply(joinback2)

In [190]:
df

,content,sentiment
0,never hurt people love lot hurt back probably ...,2
1,expect tell wrong figure ready correct cos kno...,2
2,preferred walking away fighting worth fighting...,2
3,moving forward life hard part leaving behind s...,2
4,never cry anyone life cry deserve tears deserv...,2
...,...,...
681,embarrassment anger biggest humiliation person...,0
685,strong man good wrestler strong man one contro...,0
687,man big things make angry,0
693,singing angry know punches face,0


In [191]:
#Storing the content in the X variable
X=df['content']

In [192]:
y=df.iloc[:,-1].values

---
# Standardized ML Pipeline

**Step 1** — LazyPredict: automated baseline comparison of dozens of models  
**Step 2** — PyCaret: automated final pipeline (setup → compare → finalize)


In [ ]:
# ── Feature Vectorization (for ML pipeline) ──
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

_text_series = X
_target_series = y

# Drop NaN rows
import pandas as pd
_mask = _text_series.notna() & pd.Series(_target_series).notna()
_text_clean = _text_series[_mask].astype(str)
_target_clean = np.array(pd.Series(_target_series)[_mask])

_tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
_X_vectorized = _tfidf.fit_transform(_text_clean)
_y_vectorized = _target_clean

print(f'Vectorized shape: {_X_vectorized.shape}')
print(f'Target classes: {np.unique(_y_vectorized)}')


In [ ]:
# ── STEP 1: LazyPredict — Baseline Model Comparison ──
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyClassifier
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

_X = _X_vectorized.toarray() if hasattr(_X_vectorized, 'toarray') else np.array(_X_vectorized) if not isinstance(_X_vectorized, np.ndarray) else _X_vectorized
_y = np.array(_y_vectorized).ravel()

X_train, X_test, y_train, y_test = train_test_split(_X, _y, test_size=0.2, random_state=42)

clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models, predictions = clf.fit(X_train, X_test, y_train, y_test)
print(models)

# ── Metrics Extraction ──
best_model_name = models.sort_values('Accuracy', ascending=False).index[0]
_best_row = models.loc[best_model_name]
lp_accuracy = float(_best_row.get('Accuracy', 0))
lp_f1 = float(_best_row.get('F1 Score', 0))
print(f'\n>>> Best LazyPredict model: {best_model_name}')
print(f'    Accuracy={lp_accuracy:.4f}  F1={lp_f1:.4f}')


In [ ]:
# ── STEP 2: PyCaret — Final Pipeline ──
from pycaret.classification import setup, compare_models, finalize_model, pull

# Prepare DataFrame (sample if needed for speed)
_max_rows, _max_cols = 5000, 2000
_X_sub = _X[:_max_rows, :min(_X.shape[1], _max_cols)]
_y_sub = _y[:_max_rows]

df_ml = pd.DataFrame(_X_sub, columns=[f'f{i}' for i in range(_X_sub.shape[1])])
df_ml['target'] = _y_sub

s = setup(data=df_ml, target='target', session_id=42, verbose=False)
best = compare_models(n_select=1)
pycaret_results = pull()
print(pycaret_results)

final_model = finalize_model(best)
pycaret_model_name = type(best).__name__

# Extract PyCaret metrics
_pc_best = pycaret_results.iloc[0]
pc_accuracy = float(_pc_best.get('Accuracy', 0))
pc_precision = float(_pc_best.get('Prec.', 0))
pc_recall = float(_pc_best.get('Recall', 0))
pc_f1 = float(_pc_best.get('F1', 0))

print(f'\nPyCaret Best: {pycaret_model_name}')
print(f'  Accuracy={pc_accuracy:.4f}  Precision={pc_precision:.4f}  Recall={pc_recall:.4f}  F1={pc_f1:.4f}')


---
## Model Governance — Persistence & Registry

Save trained model, feature vectorizer, metrics, and register in global project registry.


In [ ]:
import json, os
from datetime import datetime
from pathlib import Path
from joblib import dump

project_name = 'whatsapp_sentiment'
_artifacts_dir = Path('..') / 'artifacts' / project_name
_artifacts_dir.mkdir(parents=True, exist_ok=True)

# Save trained model
dump(final_model, str(_artifacts_dir / 'model.joblib'))

# Save feature vectorizer
dump(_tfidf, str(_artifacts_dir / 'vectorizer.joblib'))

# Save metrics
_metrics = {
    'best_model_lazypredict': best_model_name,
    'pycaret_model': pycaret_model_name,
    'accuracy': pc_accuracy,
    'f1': pc_f1,
    'precision': pc_precision,
    'recall': pc_recall,
    'lp_accuracy': lp_accuracy,
    'lp_f1': lp_f1,
}
with open(str(_artifacts_dir / 'metrics.json'), 'w') as f:
    json.dump(_metrics, f, indent=2)

# Update global registry
_registry_path = Path('..') / 'artifacts' / 'global_registry.json'
_registry_path.parent.mkdir(parents=True, exist_ok=True)
if _registry_path.exists():
    with open(str(_registry_path)) as f:
        _registry = json.load(f)
else:
    _registry = []
_registry = [e for e in _registry if e.get('project') != project_name]
_registry.append({
    'project': project_name,
    'best_model': best_model_name,
    'pycaret_model': pycaret_model_name,
    'accuracy': pc_accuracy,
    'timestamp': datetime.now().isoformat(),
})
with open(str(_registry_path), 'w') as f:
    json.dump(_registry, f, indent=2)

print(f'Artifacts saved to {_artifacts_dir}/')


In [ ]:
# ── Inference Function ──
def predict_text(text):
    """Run inference on a single text input."""
    vec = _tfidf.transform([text])
    return final_model.predict(vec)

print('Inference function ready: predict_text(text)')


In [ ]:
# ── Consistency Checks ──
assert final_model is not None, 'Final model was not created'
assert best_model_name is not None, 'Best model name was not captured'
assert (_artifacts_dir / 'model.joblib').exists(), 'Model file not saved'
assert (_artifacts_dir / 'metrics.json').exists(), 'Metrics file not saved'

# ── Summary ──
print('=' * 50)
print('MODEL GOVERNANCE SUMMARY')
print('=' * 50)
print(f'Project:              {project_name}')
print(f'Best Model (LP):      {best_model_name}')
print(f'Best Model (PyCaret): {pycaret_model_name}')
print(f'Accuracy:             {pc_accuracy:.4f}')
print(f'Precision:            {pc_precision:.4f}')
print(f'Recall:               {pc_recall:.4f}')
print(f'F1 Score:             {pc_f1:.4f}')
print(f'Artifacts:            {_artifacts_dir}/')
print('=' * 50)
